# 05 - Controlled Baseline: CNN + BiLSTM Model Training & Benchmark
## Tai Phake Visual Speech Recognition (VSR) Project

**Target Architecture**: Controlled **CNN + BiLSTM** (`cnn_bilstm`) baseline.

---
### Experimental Controls & Configuration:
- **Dataset**: Canonical speaker-independent partitions (`data/processed/train`, `val`, `test`).
- **Spatio-Temporal Pipeline**: Lightweight 2D CNN backbone + recurrent aggregator.
- **Temporal Standardization**: Exactly 30 frames per utterance.
- **Optimization**: Adam (lr=1e-4), max 100 epochs, Early Stopping (patience=15, monitor='val_loss').
- **Consolidated Output**: Complete evaluation metrics (accuracy in %, loss, macro-averages), classification report, confusion matrix, and training curves are organized neatly at the end of the notebook.

## 1. Imports & Environment Setup

In [11]:
# Section 1: Imports
import os
import sys
import re
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.21.0


## 2. Configuration & Dataset Loading
Loads the preprocessed manifest and builds the high-throughput `tf.data` datasets with in-memory caching.

In [12]:
# Section 2: Configuration & DataLoader
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Dataset directories: easily switchable by pointing or replacing data/processed/train, val, test
DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

RESULTS_DIR = PROJECT_ROOT / "results" / "dlib"
MODELS_DIR = PROJECT_ROOT / "results" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 11
NUM_FRAMES = 30
TARGET_WIDTH = 96
TARGET_HEIGHT = 96
CHANNELS = 3
BATCH_SIZE = 16

CNN_FEATURE_DIM = 64
RNN_HIDDEN_UNITS = 64
RNN_LAYERS = 1
DROPOUT_RATE = 0.3
LEARNING_RATE = 1e-4
MAX_EPOCHS = 100
PATIENCE = 15
RE_TRAIN = True  # Set to True to re-train all models from scratch

EXPECTED_DIGITS = [f"d{i}" for i in range(11)]
CLASS_TO_ID = {d: i for i, d in enumerate(EXPECTED_DIGITS)}
ID_TO_CLASS = {i: d for i, d in enumerate(EXPECTED_DIGITS)}

def natural_sort_key(p):
    m = re.search(r"(\d+)", Path(p).stem)
    return int(m.group(1)) if m else 0

def sample_frame_paths(fpaths, target=NUM_FRAMES):
    n = len(fpaths)
    return [str(fpaths[i]) for i in np.round(np.linspace(0, n - 1, target)).astype(int)]

def load_single_frame(path_tensor):
    raw = tf.io.read_file(path_tensor)
    img = tf.image.decode_jpeg(raw, channels=CHANNELS)
    img = tf.image.resize(img, [TARGET_HEIGHT, TARGET_WIDTH], method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([TARGET_HEIGHT, TARGET_WIDTH, CHANNELS])
    return img

def load_video_frames(paths_tensor):
    frames = tf.map_fn(
        fn=load_single_frame,
        elems=paths_tensor,
        fn_output_signature=tf.TensorSpec(shape=[TARGET_HEIGHT, TARGET_WIDTH, CHANNELS], dtype=tf.float32)
    )
    frames.set_shape([NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS])
    return frames

def index_split_utterances(split_dir, split_name):
    records = []
    speakers = sorted([d.name for d in split_dir.iterdir() if d.is_dir() and not d.name.startswith('.')])
    for spk in speakers:
        for digit in EXPECTED_DIGITS:
            ddir = split_dir / spk / digit
            if not ddir.exists():
                continue
            fpaths = sorted([f for f in ddir.glob('*.jpg')], key=natural_sort_key)
            if len(fpaths) > 0:
                records.append({
                    'speaker': spk,
                    'digit': digit,
                    'frame_paths': sample_frame_paths(fpaths, NUM_FRAMES),
                    'digit_id': CLASS_TO_ID[digit],
                    'split': split_name
                })
    return pd.DataFrame(records)

train_df = index_split_utterances(TRAIN_DIR, 'train')
val_df = index_split_utterances(VAL_DIR, 'val')
test_df = index_split_utterances(TEST_DIR, 'test')

def create_tf_dataset_from_records(df_subset, is_training=False, shuffle=True):
    all_paths = list(df_subset['frame_paths'])
    all_labels = list(df_subset['digit_id'])
    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(lambda p, l: (load_video_frames(p), tf.cast(l, tf.int32)), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache()
    if is_training and shuffle:
        ds = ds.shuffle(len(df_subset), seed=SEED)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

train_dataset = create_tf_dataset_from_records(train_df, is_training=True, shuffle=True)
val_dataset = create_tf_dataset_from_records(val_df, is_training=False, shuffle=False)
test_dataset = create_tf_dataset_from_records(test_df, is_training=False, shuffle=False)

print(f"Loaded from: Train={TRAIN_DIR}, Val={VAL_DIR}, Test={TEST_DIR}")
print(f"Speakers: {train_df['speaker'].nunique()} train, {val_df['speaker'].nunique()} val, {test_df['speaker'].nunique()} test")
print(f"Utterances: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test")
print(f"Batches: {len(train_dataset)} train | {len(val_dataset)} val | {len(test_dataset)} test")


train_dataset: 69 batches | val: 3 | test: 5


## 3. Controlled Model Architectures
We construct lightweight CNN backbones coupled with recurrent sequence aggregators (LSTM, BiLSTM, GRU, BiGRU).

In [13]:
# Section 3: Model Builder
def build_controlled_model(
    model_type="cnn_lstm",
    input_shape=(NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS),
    cnn_feature_dim=CNN_FEATURE_DIM,
    rnn_units=RNN_HIDDEN_UNITS,
    rnn_layers=RNN_LAYERS,
    dropout_rate=DROPOUT_RATE,
    num_classes=NUM_CLASSES,
    learning_rate=LEARNING_RATE
):
    inputs = tf.keras.layers.Input(shape=input_shape, name="lip_sequence_input")
    cnn = tf.keras.Sequential([
        tf.keras.layers.Conv2D(16, (3, 3), strides=(2, 2), activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(cnn_feature_dim, activation="relu")
    ], name="lightweight_cnn")

    x = tf.keras.layers.TimeDistributed(cnn, name="time_distributed_cnn")(inputs)

    m_type = model_type.lower()
    for layer_idx in range(rnn_layers):
        return_sequences = (layer_idx < rnn_layers - 1)
        if m_type == "cnn_bilstm":
            r_layer = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(rnn_units, return_sequences=return_sequences), name=f"bilstm_{layer_idx+1}")
        elif m_type == "cnn_lstm":
            r_layer = tf.keras.layers.LSTM(rnn_units, return_sequences=return_sequences, name=f"lstm_{layer_idx+1}")
        elif m_type == "cnn_bigru":
            r_layer = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(rnn_units, return_sequences=return_sequences), name=f"bigru_{layer_idx+1}")
        elif m_type == "cnn_gru":
            r_layer = tf.keras.layers.GRU(rnn_units, return_sequences=return_sequences, name=f"gru_{layer_idx+1}")
        else:
            raise ValueError(f"Unknown model_type: {model_type}")
        x = r_layer(x)

    x = tf.keras.layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="digit_classifier")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=f"vsr_{m_type}")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    return model

print("Controlled Architecture Parameters Summary:")
for m_name in ["cnn_lstm", "cnn_bilstm", "cnn_gru", "cnn_bigru"]:
    m = build_controlled_model(m_name)
    print(f"  {m_name.upper():<12}: Parameters = {m.count_params():,}")

Controlled Architecture Parameters Summary:
  CNN_LSTM    : Parameters = 61,931
  CNN_BILSTM  : Parameters = 95,659
  CNN_GRU     : Parameters = 53,867
  CNN_BIGRU   : Parameters = 79,531


## 4. Controlled Model Execution: CNN + BiLSTM
Trains the `cnn_bilstm` model with EarlyStopping, saves checkpoints, and outputs consolidated evaluation metrics on the unseen test set.

In [ ]:
# Section 4: CNN + BiLSTM (cnn_bilstm) Training and Consolidated Evaluation
m_name = "cnn_bilstm"
disp_name = "CNN + BiLSTM"

print("=" * 75)
print(f"          BASELINE EXPERIMENT: {disp_name.upper()} ({m_name})")
print("=" * 75)

model_save_dir = MODELS_DIR / m_name
model_save_dir.mkdir(parents=True, exist_ok=True)
best_model_path = model_save_dir / "best_model.keras"
hist_csv_path = model_save_dir / "training_history.csv"
metrics_json_path = model_save_dir / "metrics.json"

if not best_model_path.exists() or RE_TRAIN:
    print(f"Training {m_name} from scratch (max {MAX_EPOCHS} epochs, patience {PATIENCE})...")
    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_controlled_model(m_name)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=str(best_model_path), monitor="val_loss", save_best_only=True, verbose=0)
    ]
    t0 = time.time()
    history = model.fit(train_dataset, validation_data=val_dataset, epochs=MAX_EPOCHS, callbacks=callbacks, verbose=1)
    train_time = time.time() - t0
    print(f"Training completed in {train_time:.2f}s")
    
    hist_df = pd.DataFrame(history.history)
    hist_df.index.name = "epoch"
    hist_df.to_csv(hist_csv_path)
else:
    print(f"Loading existing best checkpoint for {m_name} from: {best_model_path.resolve()}")
    hist_df = pd.read_csv(hist_csv_path, index_col="epoch")
    train_time = 0.0

best_epoch = int(np.argmin(hist_df["val_loss"]) + 1)
best_val_loss = float(np.min(hist_df["val_loss"]))
best_val_acc = float(hist_df["val_accuracy"].iloc[best_epoch - 1])

# Evaluate on TEST set only with selected checkpoint
best_model = tf.keras.models.load_model(str(best_model_path))
total_params = int(best_model.count_params())

y_true_list, y_pred_list, y_prob_list = [], [], []
for x_b, y_b in test_dataset:
    preds = best_model.predict(x_b, verbose=0)
    y_prob_list.extend(preds)
    y_pred_list.extend(np.argmax(preds, axis=1))
    y_true_list.extend(y_b.numpy())

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)
y_prob = np.array(y_prob_list)

test_acc = float(accuracy_score(y_true, y_pred))
test_macro_prec = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
test_macro_rec = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
test_macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
per_class_prec = precision_score(y_true, y_pred, average=None, zero_division=0).tolist()
per_class_rec = recall_score(y_true, y_pred, average=None, zero_division=0).tolist()
per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0).tolist()
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

correct_count = int(np.sum(y_true == y_pred))
incorrect_count = int(np.sum(y_true != y_pred))

# Save predictions.csv
pred_df = test_df.copy()
pred_df["true_label"] = y_true
pred_df["predicted_label"] = y_pred
pred_df["predicted_digit"] = [ID_TO_CLASS[idx] for idx in y_pred]
pred_df["correct"] = (y_true == y_pred)
for cls_idx in range(NUM_CLASSES):
    pred_df[f"prob_d{cls_idx}"] = y_prob[:, cls_idx]
pred_df.to_csv(model_save_dir / "predictions.csv", index=False)

# Save metrics.json & CSV
metrics_data = {
    "model": m_name,
    "display_name": disp_name,
    "parameters_count": total_params,
    "epochs_trained": len(hist_df),
    "best_epoch": best_epoch,
    "best_val_loss": round(best_val_loss, 4),
    "best_val_accuracy": round(best_val_acc, 4),
    "test_accuracy": round(test_acc, 4),
    "test_accuracy_pct": round(test_acc * 100, 2),
    "test_macro_precision": round(test_macro_prec, 4),
    "test_macro_recall": round(test_macro_rec, 4),
    "test_macro_f1": round(test_macro_f1, 4),
    "correct_test_predictions": correct_count,
    "incorrect_test_predictions": incorrect_count,
    "per_class": {f"d{i}": {"precision": float(per_class_prec[i]), "recall": float(per_class_rec[i]), "f1": float(per_class_f1[i]), "support": int(np.sum(y_true == i))} for i in range(NUM_CLASSES)},
    "confusion_matrix": cm.tolist()
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)

# Save config.json
config_data = {
    "model_type": m_name,
    "display_name": disp_name,
    "input_shape": [NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS],
    "cnn_feature_dim": CNN_FEATURE_DIM,
    "rnn_units": RNN_HIDDEN_UNITS,
    "rnn_layers": RNN_LAYERS,
    "dropout": DROPOUT_RATE,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "optimizer": "Adam",
    "loss": "SparseCategoricalCrossentropy",
    "parameters_count": total_params
}
with open(model_save_dir / "config.json", "w") as f:
    json.dump(config_data, f, indent=2)

# Per-class summary DataFrame
df_per_class = pd.DataFrame({
    "Digit": EXPECTED_DIGITS,
    "Precision": [round(p, 4) for p in per_class_prec],
    "Recall": [round(r, 4) for r in per_class_rec],
    "F1-Score": [round(f, 4) for f in per_class_f1],
    "Support": [int(np.sum(y_true == i)) for i in range(NUM_CLASSES)]
}).set_index("Digit")
df_per_class.to_csv(model_save_dir / "per_class_metrics.csv")

# ==============================================================================
# 5. CONSOLIDATED ALL-IN-ONE OUTPUT RESULTS (EASY TO READ AT A GLANCE)
# ==============================================================================
print("\n" + "=" * 80)
print(f"                 FINAL EVALUATION SUMMARY: {disp_name.upper()} ({m_name})")
print("=" * 80)
print(f"  Test Accuracy:           {test_acc * 100:.2f}%  ({correct_count}/{len(y_true)} correct)")
print(f"  Validation Accuracy:     {best_val_acc * 100:.2f}%  (Best Epoch: {best_epoch})")
print(f"  Validation Loss:         {best_val_loss:.4f}")
print(f"  Macro Precision:         {test_macro_prec:.4f}")
print(f"  Macro Recall:            {test_macro_rec:.4f}")
print(f"  Macro F1-Score:          {test_macro_f1:.4f}")
print(f"  Total Parameters:        {total_params:,}")
if train_time > 0:
    print(f"  Training Duration:       {train_time:.2f}s")
print("-" * 80)
print("Per-Class Metrics (Unseen Test Set):")
display(df_per_class)
print("=" * 80)

# 1. Training & Validation History Trajectories Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_range = range(1, len(hist_df) + 1)

axes[0].plot(epochs_range, hist_df["loss"], label="Train Loss", color="royalblue", lw=2)
axes[0].plot(epochs_range, hist_df["val_loss"], label="Val Loss", color="crimson", lw=2)
axes[0].axvline(best_epoch, color="gray", linestyle="--", label=f"Best Epoch ({best_epoch})")
axes[0].set_title(f"{disp_name}: Loss Trajectory", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

axes[1].plot(epochs_range, hist_df["accuracy"] * 100, label="Train Accuracy (%)", color="royalblue", lw=2)
axes[1].plot(epochs_range, hist_df["val_accuracy"] * 100, label="Val Accuracy (%)", color="crimson", lw=2)
axes[1].axvline(best_epoch, color="gray", linestyle="--", label=f"Best Epoch ({best_epoch})")
axes[1].set_title(f"{disp_name}: Accuracy Trajectory (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.savefig(model_save_dir / "training_curves.png", dpi=150)
plt.show()

# 2. Confusion Matrix Heatmap
df_cm = pd.DataFrame(cm, index=EXPECTED_DIGITS, columns=EXPECTED_DIGITS)
df_cm.to_csv(model_save_dir / "confusion_matrix.csv")

plt.figure(figsize=(8.5, 6.5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", cbar=True, square=True)
plt.title(f"Confusion Matrix: {disp_name} (Test Accuracy: {test_acc * 100:.2f}%)", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Class", fontsize=11)
plt.ylabel("True Class", fontsize=11)
plt.tight_layout()
plt.savefig(model_save_dir / "confusion_matrix.png", dpi=150)
plt.show()

print(f"\nAll model artifacts saved to: {model_save_dir.resolve()}")
